**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Capstone: Build a Full System

Not new theory — proof that the curriculum *composes*. One pipeline, end to end: IQ capture → detection (matched filter + CFAR) → tracking (Kalman) → classification (CNN) → results database. Every stage is a workshop you've taken; here they hold hands. Runs on synthetic IQ so it executes anywhere; swap in an [RTL-SDR capture](../Intro_SDR/Software_Defined_Radio.ipynb) and nothing else changes.

## 1. Pre-requisites

The whole curriculum, honestly — minimally: [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb), [Kalman](../Intro_Time_Series/Intro_AdFilt_KF.ipynb), [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb), [Databases](../Intro_Host_Prog/Intro_Databases/Intro_Databases.ipynb).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import sqlite3
import matplotlib.pyplot as plt
from scipy import signal as sig
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *The Scenario & the Signal Generator* (~30 min)
**Goal:** an emitter moves through a noisy band, transmitting bursts; we build the world to be monitored.
**Feeds into:** Session 2 (detection).

---

## 2. The World

💡 **Intuition.** System design starts with the truth you'll grade yourself against. Our world: an emitter drifts across the band (a slowly-moving carrier frequency), transmitting short bursts of one of three modulation types, buried in noise. The pipeline must find the bursts, track the drift, and identify the modulation — and because we built the world, every stage gets an oracle.

In [2]:
fs = 100_000
DUR, BURST, GAP = 4.0, 0.02, 0.05
n_samp = int(DUR*fs)
t_all = np.arange(n_samp)/fs

# emitter truth: carrier random-walks; burst type fixed per emitter
f_true = 20_000 + np.cumsum(rng.normal(0, 30, n_samp))          # drifting carrier [Hz]
MOD = "chirp"                                                    # this emitter's fingerprint
burst_starts = np.arange(0.1, DUR-0.1, BURST+GAP)

iq = (rng.standard_normal(n_samp) + 1j*rng.standard_normal(n_samp)) * np.sqrt(0.5)  # noise floor
n_burst = int(BURST*fs)
for bs in burst_starts:
    i0 = int(bs*fs)
    tt = np.arange(n_burst)/fs
    fc = f_true[i0]
    if MOD == "chirp":  base = sig.chirp(tt, 0, BURST, 3000)
    env = np.exp(2j*np.pi*fc*tt) * base
    iq[i0:i0+n_burst] += 4.0 * env
print(f"{len(burst_starts)} bursts planted; carrier wanders {f_true.min()/1e3:.1f} → {f_true.max()/1e3:.1f} kHz")

55 bursts planted; carrier wanders 13.5 → 35.1 kHz


---
### 🕐 Session 2 of 4 — *Detection: Energy → CFAR* (~35 min)
**Goal:** find the bursts in time-frequency; CFAR keeps false alarms honest.
**Builds on:** [Statistical SP](../Intro_DSP/Statistical_Signal_Processing.ipynb) S4; [Radar](../Intro_DSP/Radar_Signal_Processing.ipynb) S3. &nbsp; **Feeds into:** Session 3 (tracking).

---

In [3]:
f_stft, t_stft, Z = sig.stft(iq, fs=fs, nperseg=1024, noverlap=768, return_onesided=False)
P = np.abs(Z)**2
# collapse to a detection statistic per frame: max power over frequency
stat = P.max(0)
# CA-CFAR along time (from the Radar workshop, verbatim idea)
n_ref, n_guard, scale = 24, 4, 7.0
th = np.full_like(stat, np.inf)
for i in range(n_ref+n_guard, len(stat)-n_ref-n_guard):
    ref = np.r_[stat[i-n_ref-n_guard:i-n_guard], stat[i+n_guard+1:i+n_guard+1+n_ref]]
    th[i] = scale*np.median(ref)                       # median-CFAR: robust to burst pollution
det = stat > th
# group consecutive detections into events; record their peak frequency
events = []
i = 0
while i < len(det):
    if det[i]:
        j = i
        while j < len(det) and det[j]: j += 1
        k = i + np.argmax(stat[i:j])
        f_peak = f_stft[np.argmax(P[:, k])]
        events.append((t_stft[k], f_peak % fs))
        i = j
    else: i += 1
events = [(tt_, f_ if f_ < fs/2 else f_ - fs) for tt_, f_ in events]

hits = sum(any(abs(tt_ - (bs+BURST/2)) < 0.03 for tt_, _ in events) for bs in burst_starts)
print(f"planted bursts: {len(burst_starts)}   detected events: {len(events)}   planted-burst hit rate: {hits/len(burst_starts):.0%}")

planted bursts: 55   detected events: 55   planted-burst hit rate: 100%


---
### 🕐 Session 3 of 4 — *Tracking the Drift: Kalman* (~35 min)
**Goal:** the detections are noisy frequency snapshots; a Kalman filter strings them into a track.
**Builds on:** [Kalman workshop](../Intro_Time_Series/Intro_AdFilt_KF.ipynb). &nbsp; **Feeds into:** Session 4 (classification & the database).

---

In [4]:
# constant-velocity Kalman on (frequency, drift-rate), fed by the detected events
meas = sorted(events)
dt_meas = np.diff([m[0] for m in meas]).mean()
F = np.array([[1, dt_meas], [0, 1]])
Hm = np.array([[1.0, 0.0]])
Q = np.diag([10.0, 3000.0]); R = np.array([[80_000.0]])
x_kf = np.array([meas[0][1], 0.0]); Pk = np.diag([1e6, 1e6])
track = []
for tt_, f_meas in meas:
    x_kf = F @ x_kf; Pk = F @ Pk @ F.T + Q
    S = Hm @ Pk @ Hm.T + R
    K = Pk @ Hm.T / S
    x_kf = x_kf + (K * (f_meas - Hm @ x_kf)).ravel()
    Pk = (np.eye(2) - K @ Hm) @ Pk
    track.append((tt_, x_kf[0]))

track_t = np.array([a for a, _ in track]); track_f = np.array([b for _, b in track])
truth_at = np.interp(track_t, t_all, f_true)
rmse_raw = np.sqrt(np.mean((np.array([f_ for _, f_ in meas]) - truth_at)**2))
rmse_kf = np.sqrt(np.mean((track_f - truth_at)**2))
plt.figure(figsize=(9, 2.8))
plt.plot(t_all[::100], f_true[::100]/1e3, "k--", linewidth=1, label="true carrier")
plt.plot([m[0] for m in meas], [m[1]/1e3 for m in meas], ".", alpha=0.5, label="detections")
plt.plot(track_t, track_f/1e3, "C1", label="Kalman track")
plt.legend(fontsize=8); plt.xlabel("time [s]"); plt.ylabel("kHz")
plt.title(f"raw detections RMSE {rmse_raw:.0f} Hz → Kalman track RMSE {rmse_kf:.0f} Hz")
plt.tight_layout(); plt.show()

/tmp/ipykernel_321604/2008688132.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 4 of 4 — *Classification & the Ledger* (~40 min)
**Goal:** a CNN identifies each burst's modulation; every verdict lands in a queryable database.
**Builds on:** [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb); [Databases](../Intro_Host_Prog/Intro_Databases/Intro_Databases.ipynb).

---

In [5]:
# train the classifier on synthetic bursts of the 3 candidate types (baseband spectrograms)
def burst_example(kind):
    tt = np.arange(n_burst)/fs
    df = rng.normal(0, 150)                                  # residual tracking error, like the pipeline's
    if kind == 0: base = sig.chirp(tt, 0, BURST, 3000)
    elif kind == 1: base = np.sign(np.sin(2*np.pi*400*tt))*np.cos(2*np.pi*1500*tt)   # BPSK-ish
    else: base = np.cos(2*np.pi*(1500 + 800*np.sin(2*np.pi*60*tt))*tt)               # FM tone
    x = 4.0*base*np.cos(2*np.pi*df*tt) + 1.0*rng.standard_normal(n_burst)   # match pipeline amplitude, noise, AND residual offset
    _, _, S = sig.stft(x, fs=fs, nperseg=128)
    S = np.log1p(np.abs(S))[:32, :14]
    return (S - S.mean())/(S.std()+1e-6)

Xc = torch.tensor(np.stack([burst_example(k % 3) for k in range(1200)]), dtype=torch.float32)[:, None]
yc = torch.tensor([k % 3 for k in range(1200)])
cnn = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(8*16*7, 3))
opt = torch.optim.Adam(cnn.parameters(), lr=2e-3)
for ep in range(6):
    for i in range(0, 1000, 64):
        opt.zero_grad(); nn.functional.cross_entropy(cnn(Xc[i:i+64]), yc[i:i+64]).backward(); opt.step()
acc = (cnn(Xc[1000:]).argmax(1) == yc[1000:]).float().mean()
print(f"classifier holdout accuracy: {acc:.0%}")

# run it on the DETECTED bursts (mix each event down to baseband using the KALMAN track)
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE detections (t REAL, f_hz REAL, verdict TEXT, confidence REAL)")
names = ["chirp", "bpsk", "fm"]
votes = []
with torch.no_grad():
    for tt_, f_tr in track:
        i0 = max(0, int(tt_*fs) - n_burst//2)
        seg = iq[i0:i0+n_burst]
        if len(seg) < n_burst: continue
        bb = np.real(seg * np.exp(-2j*np.pi*f_tr*np.arange(len(seg))/fs))   # Kalman-guided downmix
        _, _, S = sig.stft(bb, fs=fs, nperseg=128)
        S = np.log1p(np.abs(S))[:32, :14]; S = (S - S.mean())/(S.std()+1e-6)
        logits = cnn(torch.tensor(S, dtype=torch.float32)[None, None])
        p = torch.softmax(logits, 1)[0]
        votes.append(int(p.argmax()))
        db.execute("INSERT INTO detections VALUES (?,?,?,?)", (tt_, f_tr, names[p.argmax()], float(p.max())))
db.commit()

print("\nthe ledger answers questions (the Databases workshop, cashing in):")
for row in db.execute("SELECT verdict, COUNT(*), AVG(confidence) FROM detections GROUP BY verdict ORDER BY 2 DESC"):
    print(f"  {row[0]:6s}: {row[1]:3d} detections, mean confidence {row[2]:.2f}")
maj = names[max(set(votes), key=votes.count)]
print(f"\nsystem verdict on the emitter: '{maj}'   (planted truth: 'chirp')")

classifier holdout accuracy: 100%

the ledger answers questions (the Databases workshop, cashing in):
  chirp :  32 detections, mean confidence 0.69
  bpsk  :  19 detections, mean confidence 0.78
  fm    :   4 detections, mean confidence 0.50

system verdict on the emitter: 'chirp'   (planted truth: 'chirp')


## 3. Conclusion

Detection found the bursts (hit rate printed), the Kalman track cut the frequency error and *steered the downmixer*, the CNN identified the modulation, and SQL holds the evidence. No stage is new; the composition is the achievement — and every interface between stages is a place your own projects can swap in smarter pieces.

**The real-hardware version:** replace Session 1 with an [RTL-SDR capture](../Intro_SDR/Software_Defined_Radio.ipynb); budget the pipeline with [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb); containerize it with [Containers](../Intro_Host_Prog/Intro_Containers/Intro_Containers.ipynb). That's a senior-project-grade system, from parts you already own.

---
**This is the final playlist.** Where next is yours.